<a href="https://colab.research.google.com/github/fredi-star/Kiosk-Wetter-staffing-prediction-try/blob/main/oldKioskProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🌤️ Kiosk Staffing Predictor: Analyse & Modellierung

### ⚠️ Projekt-Disclaimer
*Dieses Projekt ist eine Rekonstruktion eines realen Business Cases. Aus Datenschutzgründen wird ein **synthetischer Datensatz** verwendet, der die statistischen Eigenschaften der Originaldaten simuliert. Dieses Notebook dient zur Demonstration der methodischen Vorgehensweise (Data Cleaning, Feature Engineering, Predictive Modeling) und der Coding-Standards. Bei der Erstellung des Codes und der Dokumentation wurden KI-Tools zur Optimierung und Strukturierung unterstützend eingesetzt.*

---

### 🎯 Zielsetzung
Entwicklung eines regelbasierten Modells zur Vorhersage von Personalbedarf basierend auf historischen Verkaufszahlen und Wetterdaten. Ziel ist es, Tage mit hohem Umsatzpotenzial frühzeitig zu erkennen, um Unterbesetzung zu vermeiden.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import holidays

# Plotting Style für professionelle Grafiken
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

In [ ]:
def load_and_clean_data(filepath):
    """
    Lädt die Daten, konvertiert Datumsformate und bereinigt Anomalien sowie Feiertage.
    """
    # 1. Daten laden
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(f"Datei '{filepath}' nicht gefunden. Bitte Pfad prüfen.")
        return None

    # 2. Datentypen und Sortierung
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(by='date').reset_index(drop=True)

    # 3. Feiertage ausschließen (Beispiel: Hamburg)
    # Dies ist wichtig, da Feiertage andere Umsatzmuster aufweisen
    hh_holidays = holidays.DE(prov='HH', years=df['date'].dt.year.unique())
    df['is_holiday'] = df['date'].isin(hh_holidays)

    # Filter: Keine Feiertage und Mindestumsatz (um geschlossene Tage zu entfernen)
    df_clean = df[(~df['is_holiday']) & (df['sales_eur'] > 300)].copy()

    # 4. Anomalie-Erkennung
    # Wir nutzen einen gleitenden 7-Tage-Durchschnitt zur Erkennung extremer Ausreißer
    df_clean['rolling_mean_7d'] = df_clean['sales_eur'].rolling(window=7).mean()

    # Threshold: 70% Abweichung vom Wochendurchschnitt gilt als Anomalie (z.B. Events)
    anomaly_threshold = 0.70
    df_clean['is_anomaly'] = abs(df_clean['sales_eur'] - df_clean['rolling_mean_7d']) > (anomaly_deviation_threshold * df_clean['rolling_mean_7d'])

    # Anomalien entfernen
    df_final = df_clean[~df_clean['is_anomaly']].copy()

    print(f"Daten geladen: {len(df)} Zeilen.")
    print(f"Nach Bereinigung (Feiertage/Anomalien): {len(df_final)} Zeilen.")

    return df_final

# Dateipfad hier anpassen
# df = load_and_clean_data('kiosk_sales_data.csv')

# HINWEIS : Da wir keine echte CSV haben, generieren wir hier Dummy-Daten
# um die Funktionalität des Notebooks zu zeigen.
dates = pd.date_range(start='2024-01-01', periods=100)
df = pd.DataFrame({
    'date': dates,
    'sales_eur': np.random.normal(4500, 800, 100), # Normalverteilter Umsatz
    'temp_C': np.random.normal(15, 5, 100),       # Temperatur
    'extra_staff': np.random.choice([0, 1], 100, p=[0.7, 0.3]) # Historische Personalentscheidung
})
# Manuelle Anpassung für Korrelation: Mehr Umsatz bei höherer Temp
df['sales_eur'] += df['temp_C'] * 100

# Anwenden der Cleaning Pipeline auf die (Dummy-)Daten
# Wir simulieren hier den Aufruf, als käme es aus einer CSV
df_clean = df.copy()
df_clean['rolling_mean_7d'] = df_clean['sales_eur'].rolling(window=7).mean()
df_clean = df_clean.dropna() # Drop first 7 NaN rows

In [1]:
def engineer_features(df):
    """
    Erstellt Features für das Modell:
    1. Lagged Moving Average (Wissen von Gestern)
    2. Temperaturänderung (Heute vs Gestern)
    3. Residuals (Umsatzabweichung vom Trend)
    """
    df_eng = df.copy()

    # 1. 7-Tage Durchschnitt von GESTERN (Shift 1), da wir den heutigen Umsatz noch nicht kennen
    df_eng['sales_7d_ma_lagged'] = df_eng['sales_eur'].rolling(window=7).mean().shift(1)

    # 2. Temperaturänderung (Differenz zum Vortag)
    df_eng['temp_change'] = df_eng['temp_C'].diff()

    # 3. Residuals berechnen (Wie stark weicht der Umsatz vom erwarteten Trend ab?)
    # Dies ist unsere Zielvariable für die Regression
    df_eng['residual'] = df_eng['sales_eur'] - df_eng['sales_7d_ma_lagged']

    # Entfernen von NaN Werten, die durch Shifting entstanden sind
    return df_eng.dropna()

df_features = engineer_features(df_clean)
display(df_features[['date', 'sales_eur', 'temp_change', 'residual']].head())

NameError: name 'df_clean' is not defined

In [ ]:
# Berechnung der Korrelation
corr = df_features['temp_change'].corr(df_features['residual'])

# Visualisierung
plt.figure(figsize=(10, 6))
sns.regplot(x='temp_change', y='residual', data=df_features,
            scatter_kws={'alpha':0.6}, line_kws={'color':'red', 'label':'Linear Fit'})

plt.title(f'Einfluss der Temperaturänderung auf Umsatzabweichung (Corr: {corr:.2f})', fontsize=14)
plt.xlabel('Temperaturänderung (°C)')
plt.ylabel('Umsatz-Residual (EUR)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Definition der Variablen
X = df_features['temp_change']
y = df_features['residual']

# Hinzufügen einer Konstante (Intercept) für OLS
X = sm.add_constant(X)

# Modell Training
model = sm.OLS(y, X).fit()

# Ausgabe der wichtigsten Parameter
print("--- Modell Parameter ---")
print(f"Intercept (Basis-Abweichung): {model.params['const']:.2f} EUR")
print(f"Koeffizient (Pro 1°C Änderung): {model.params['temp_change']:.2f} EUR")
print(f"R-Squared: {model.rsquared:.3f}")

In [ ]:
def run_backtest(df, model, base_threshold):
    """
    Testet verschiedene Schwellenwerte für die Personalplanung.
    """
    results = []

    # Vorhersage des Umsatzes für alle Tage basierend auf dem Modell
    # Formel: Gestern_MA + Intercept + (Temp_Change * Coeff)
    intercept = model.params['const']
    coeff = model.params['temp_change']

    predicted_sales = (
        df['sales_7d_ma_lagged'] +
        intercept +
        (df['temp_change'] * coeff)
    )

    # Testen verschiedener Sensitivitäten (+/- X% vom Durchschnitt)
    variations = [-0.15, 0.0, 0.15, 0.30]

    for v in variations:
        threshold = base_threshold * (1 + v)

        # Entscheidung: Brauchen wir Personal? (Vorhersage > Schwellenwert)
        pred_staff = predicted_sales >= threshold
        actual_staff = df['extra_staff'].astype(bool)

        # Metriken berechnen
        tp = ((pred_staff) & (actual_staff)).sum()
        fp = ((pred_staff) & (~actual_staff)).sum()
        fn = ((~pred_staff) & (actual_staff)).sum()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        results.append({
            'Threshold Variation': f"{v*100:+.0f}%",
            'Abs. Threshold': f"{threshold:.0f} €",
            'Precision': f"{precision:.2f}",
            'Recall': f"{recall:.2f}",
            'True Positives': tp
        })

    return pd.DataFrame(results)

# Durchschnittsumsatz an Tagen mit Extra-Personal als Basis
avg_sales_staff_days = df_features[df_features['extra_staff'] == 1]['sales_eur'].mean()

# Backtest ausführen
evaluation_df = run_backtest(df_features, model, avg_sales_staff_days)

print(f"Basis-Schwellenwert (Historischer Ø): {avg_sales_staff_days:.2f} EUR")
display(evaluation_df)

In [ ]:
## 📝 Fazit & Business Empfehlung

### Ergebnisse
Die Analyse zeigt eine nachweisbare Korrelation zwischen positiven Temperaturänderungen und Umsatzanstiegen, die über den gleitenden Wochendurchschnitt hinausgehen. Das Regressionsmodell konnte diesen Zusammenhang quantifizieren.

### Herausforderung beim Backtesting
Beim Testen der Vorhersageregel (Backtest) zeigte sich ein klassischer **Trade-off zwischen Precision und Recall**:
1.  **Hoher Schwellenwert:** Führt zu hoher Präzision (wenig unnötige Personalkosten), aber niedrigem Recall (wir verpassen umsatzstarke Tage und verlieren Umsatz durch Unterbesetzung).
2.  **Niedriger Schwellenwert:** Fängt fast alle umsatzstarken Tage ab (hoher Recall), führt aber oft zu Überbesetzung an Tagen, die schwächer ausfallen als vorhergesagt.

### Empfehlung
Das aktuelle Modell auf Basis von nur zwei Faktoren (Trend + Wetter) ist besser als reines Raten, reicht aber für eine vollautomatische Personalplanung noch nicht aus.
**Nächste Schritte:**
* Integration von Wochentags-Faktoren (Wochenende vs. Woche).
* Berücksichtigung von lokalen Events (außerhalb von Feiertagen).
* Implementierung eines "Human-in-the-loop" Ansatzes: Das Modell liefert eine Empfehlung, der Schichtleiter trifft die finale Entscheidung.